# 研究架構

::: {.callout-tip}
投影片操作：**Alt + 點擊** 可縮放任何圖片/表格；`O` 鍵總覽、`F` 全螢幕。
:::

本研究依配對交易的兩階段結構，分別檢驗兩項改良，並檢驗兩者的組合：

| | 檢定的對象 | 對照設計 | 方向 | 校正後 |
| :--- | :--- | :--- | :--- | :---: |
| **分組層** | 分組施加的限制是否提升配對品質 | 5 分組 × 3 排序（分組為唯一變因） | **9/9 偏向 GICS** | 0/9 |
| **交易層** | 學習法選擇門檻是否優於固定門檻 | 5 配對底（交易端為唯一變因） | **5/5 為正** | 0/5 |
| **兩層組合** | 完整系統 vs 傳統基準 | 3 組（排序已對齊） | **全期 3/3 為負** | 0/3 |

::: {.callout-important}
**三項檢定經 BH-FDR 校正後皆無顯著。方向的一致性是最可靠的訊號。**

完整系統**劣於**傳統基準——分群層的損害超過交易端的貢獻。
:::

**共同控制條件**：混合特徵（報酬 PCA ⊕ PIT 基本面 ⊕ GICS one-hot）、
共整合篩選（ADF 0.05，**EG 臨界值**）、S&P 500 歷史成分股 2001–2025、
形成期 252 日 / 交易期 126 日 / 滾動 21 日、交易成本單邊 0.29%。

# 方法論：形成期四層架構

策略 = 四個可獨立替換的層之組態，由中性組裝器依參數組裝：

| 層 | 職責 | 本研究採用的選項 |
| :--- | :--- | :--- |
| **特徵** | 個股 → 特徵向量 | 報酬 PCA 因子載荷 ⊕ 基本面 ⊕ 產業 one-hot |
| **分組** | 特徵 → 配對搜尋空間 | GICS 產業／HDBSCAN／Agglomerative／K-means |
| **排序** | 群內配對 → 優先序 | SSD／DTW／SSD-DTW-PCA |
| **篩選** | 統計檢定淘汰 | ADF 共整合 + OU 半衰期 + Hurst |

交易期則為 **Z-Score 狀態機**（規則型基準）或 **DL-THR 門檻選擇式**（Kim & Kim 2019 風格）。

> 此架構使「分組」「排序」「交易端」皆可作為單變因替換，是兩層能乾淨對照的前提。
> 實作正確性以數值回歸測試保證：組裝器復現原生策略的結果為**逐位元相同**。


# 參考文獻與方法對應

## 分群方法
> Campello, Moulavi & Sander (2013). Density-based clustering based on hierarchical density estimates. *PAKDD*.
> Ward (1963). Hierarchical grouping to optimize an objective function. *JASA*, **58**(301).
> MacQueen (1967). Some methods for classification and analysis of multivariate observations.

HDBSCAN（自動群數、噪音標記）、Agglomerative（dendrogram 分位數校準）、
K-means（群數對齊同期 Agglomerative，使量級可比）。

## 排序準則
> Gatev, Goetzmann & Rouwenhorst (2006). Pairs trading. *RFS*, **19**(3).　📄 `ref/2006-...pdf`
> 許鈞翔 (2025)。最小距離法結合動態時間校正之配對交易研究。　📄 `ref/2025-...pdf`

SSD（同步距離）、DTW（Sakoe-Chiba 時間扭曲）、SSD-DTW-PCA（兩距離 PCA 融合）。

## 特徵與交易端
> Avellaneda & Lee (2010). Statistical arbitrage in the U.S. equities market. *Quantitative Finance*, **10**(7).　📄
> Hong & Hwang (2021). In search of pairs using firm fundamentals. *EJF*, **29**(5).　📄
> Kim & Kim (2019). Optimizing the pairs-trading strategy using DRL with trading and stop-loss boundaries. *Complexity*.　📄

報酬 PCA 因子載荷（Avellaneda & Lee）、基本面配對（Hong & Hwang）、
DL-THR 門檻選擇式交易（Kim & Kim）。


# 分組層：機器學習分群 vs 傳統產業分類

**設計**：固定特徵、篩選、Z-Score 交易端；變動分組方法 × 排序準則。

## 各格網格最佳值（最佳年化 / 最佳 Sharpe）

| 分組＼排序 | SSD | DTW | SSD-DTW-PCA |
| :--- | :---: | :---: | :---: |
| **GICS（傳統）** | 1.66% / 0.20 | 1.10% / 0.18 | 1.65% / **0.35** |
| **HDBSCAN** | 1.29% / 0.17 | 0.89% / 0.15 | **1.70%** / 0.23 |
| **Agglomerative** | **1.77% / 0.26** | 0.60% / 0.11 | 1.13% / 0.18 |
| **K-means** | 1.31% / 0.20 | −0.06% / 0.03 | 0.67% / 0.12 |

若僅看此表，會得到「Agglomerative × SSD 優於 GICS × SSD（1.77% vs 1.66%）」的印象。
然而兩者皆為 15 格中的最大值，該差距未經檢定。


## 分組層的假設檢定：方向全部偏向 GICS

逐日報酬差 $\Delta r_t = r_{ML,t} - r_{GICS,t}$（15 格等權組合，6,287 交易日），
循環 block bootstrap（$L$=126）。9 組比較以 Benjamini-Hochberg 控制 FDR。

| 分組＼排序 | SSD | DTW | SSD-DTW-PCA |
| :--- | :---: | :---: | :---: |
| HDBSCAN | −0.715 (p=0.068) | −0.387 (p=0.335) | −0.721 (p=0.061) |
| Agglomerative | −0.447 (p=0.322) | −0.229 (p=0.607) | −0.451 (p=0.281) |
| K-means | −0.913 (p=0.080) | −0.900 (p=0.118) | −1.000 (p=0.065) |

*年化報酬差（百分點，ML − GICS）；原始 $p$ 值。**BH 校正後最小 $p$ = 0.179***

**9 組方向全部偏向 GICS，校正後 0 組顯著。**
（9 組彼此不獨立——每三組共用同一 GICS 臂，故不對方向一致性施加正式檢定。）

::: {.callout-important}
### 「不顯著」不等於「兩者相當」

最小可偵測效果 **1.259 pp**，**大於** GICS 參照臂自身的等權年化（+0.419%）。
對 0.3 pp 的真實效果，檢定力僅 **10%**。

→ 資料無法排除「資料驅動分群**顯著較差**」，其程度遠甚於無法排除「兩者相當」。
:::

## 分組層為何是淨損害：期末強制平倉機制

| 分組方式 | 排除標的 | **強平率** | 平均 Sharpe |
| :--- | ---: | ---: | ---: |
| **不分組** | **0%** | **36.8%** | **+0.081** |
| GICS 產業 | 15.1% | 39.6% | +0.024 |
| HDBSCAN | 26.4% | 40.5% | −0.116 |
| Agglomerative | 35.3% | 41.5% | −0.065 |
| K-means | 42.0% | 43.1% | −0.249 |

**損益結構**（489 個 Z-Score 配置）：逐筆勝率 **0.578**、獲利因子 0.969、
期末強平佔進場 **0.418**——多數交易收斂獲利，但被少數大額虧損吃光。

| 層級 | 強平率與 Sharpe 的相關 |
| :--- | :--- |
| 逐臂（15 臂） | Pearson $r$ = **−0.895**（$p$<0.0001） |
| 逐配置內、跨 15 臂 | **15/15 為負**，中位 $r$ = −0.801 |

::: {.callout-warning}
全樣本混合（n=225）的相關為 **+0.201**，符號相反——停損維度造成的 Simpson 悖論。
**不可引用混合值。**
:::

## 粒度掃描：分群演算法從來不是關鍵變因

前頁的兩項代價在原設計中互相混淆——三種分群演算法產生的群數本就不同，
無法斷定落後源於「演算法」還是「粒度」。

**受控實驗**：固定 Agglomerative 演算法、特徵、SSD 排序、篩選與交易端，
唯一變因為切割門檻分位 $q$（分位越高 → 群越少越大 → 候選池越大）。

| 門檻分位 | 期均配對數 | 跨產業% | 最佳年化 | 網格均 Sharpe | vs GICS $p$ |
| :---: | :---: | :---: | :---: | :---: | :---: |
| 50 | 6.2 | 9.7% | −0.13% | −0.237 | 0.727 |
| **60** | 8.7 | 10.3% | **1.92%** | **−0.176** | 0.102 |
| 75（基準） | 11.5 | 10.7% | 1.77% | −0.240 | 0.586 |
| 90 | **15.0** | 31.9% | 0.40% | −0.297 | 0.140 |
| 95 | 16.8 | **50.0%** | 0.38% | −0.249 | 0.772 |
| **GICS** | **15.0** | **0.0%** | 1.66% | −0.261 | — |

**發現一：粒度的影響遠大於演算法的影響。**
單一參數即使最佳年化自 −0.13% 變動至 1.92%（幅度 2.05pp）；
相較之下，固定粒度下三種分群演算法的差距僅 0.48pp（AGG 1.77 / KM 1.31 / HDB 1.29）。
**分組層 原本比較的「分群演算法」，並非決定配對品質的主要變因。**
（五組門檻分位與 GICS 的對照 $p$ = 0.10 ~ 0.77，**無一顯著**——
本掃描刻畫的是形成期結構隨粒度的變化，不是已證實的績效差異。）

**發現二：關係為倒 U 形，非單調。** 峰值在 $q$ = 60–75，兩端皆劣化。
兩個機制反向作用——
粒度過細（$q$=50）候選池僅 6.2 對，填不滿目標而被迫接受劣質配對；
粒度過粗（$q$≥90）池雖變大，但跨產業比例自 10% 暴增至 32–50%。
候選池大小與跨產業比例的 Spearman $ho = +1.000$：**兩者在 ML 分群下無法解耦。**


## 篩選層與分組層的交互作用

文獻多將「分群 + 共整合篩選」並用，未討論兩者的交互作用。排序固定 SSD：

| 分組 | 統計篩選 | 產業 one-hot | 平均 Sharpe | 等權年化 |
| :--- | :---: | :---: | ---: | ---: |
| 不分組 | ADF | — | **+0.099** | **+0.715%** |
| 不分組 | 無 | — | +0.028 | +0.190% |
| Agglomerative | ADF | 1.0 | −0.092 | −0.112% |
| Agglomerative | 無 | 1.0 | −0.078 | −0.296% |
| Agglomerative | ADF | 0 | −0.178 | −0.107% |
| Agglomerative | 無 | 0 | −0.009 | **+0.306%** |

::: {.callout-important}
**篩選層的價值取決於候選池規模。**

不分組時施加篩選：+0.190% → **+0.715%**（有益）
分群且無產業先驗時施加篩選：+0.306% → **−0.107%**（有害）
:::

> 四項條件全開（分群 + 篩選 + 產業先驗，即多數文獻的預設）為 **−0.112%**，
> 比表中任一格都差。

::: {.aside}
描述性比較，未施加統計檢定；最大差距 0.82 pp 低於 1.26 pp 的偵測門檻。
:::

## 交易機制的逐步歸因：方向一致，但仍不顯著

形成期實作差異被排除後，剩餘殘差之一為**交易機制**。Han et al. 押的是
**群內短期反轉**（相似兩檔上月分開、下月收斂），本研究押的是
**共整合均值回歸**（價差歷史平穩、現在偏離 2σ）——訊號、持有期、退出條件全不同。

四項差異逐步施加以維持單變因（直接完整復刻會一次改四個變因）：

| 步驟 | 該步改變 | 全網格等權年化% | Top1/SL0年化% |
| :--- | :--- | ---: | ---: |
| 起點 | SSD 距離 + OLS-β + $z$>2 / 126 日 | −0.373 | −0.909 |
| ② | β 改 1 等金額 | −0.177 | −0.370 |
| ③ | 選對準則改月報酬發散 | −0.092 | +0.207 |
| ④ | 21 日窗 + 發散即建倉 + 持有至期末 | −0.094 | **+1.369** |

::: {.callout-important}

### 每一步都改善，但沒有一步顯著

Top1 口徑單調改善（−0.909% → +1.369%，計 +2.28pp），惟逐步對照校正前最小
$p$ = 0.055、BH 校正後無一顯著；總效果等權 +0.28pp（$p$=0.923）、
Top1 +2.28pp（$p$=0.705）。`entry_z`=0 搭配持有至期末使單一配對日報酬波動極大，
檢定力極低。**完整復刻交易端後等權年化仍為 −0.094%**，離原文 24.8% 差兩個數量級。

:::

**附帶發現：排序準則本身亦為產業偏誤的來源。**
`HAN3-REV` 的跨產業配對比例 19.7%，而分群設定完全相同、僅排序準則不同的
`AGG-SSD-NF` 僅 6.8%。SSD 距離偏好同產業配對——同業股票的歷史價格路徑天然更接近。
此點在排序固定為 SSD 的因子設計中無法觀察。

> **綜合**：形成期實作差異與交易機制差異皆已檢驗，**皆非分組層未顯示效果的原因**。
> 剩餘差距歸因於兩項**資料可得性限制**而非設計選擇——分群特徵 7 維連續
> （原文 48 動量因子 + 78 公司特徵）、母體 S&P 500（原文 CRSP 全市場）。


## 強平的配對只是「尚未回歸」嗎？

對 **4,738 筆**期末強制平倉的交易，往後追蹤 126 個交易日（一個完整交易期）：

| 追蹤期 | 21 日 | 42 日 | 63 日 | **126 日** |
| :--- | ---: | ---: | ---: | ---: |
| 累計回歸比例 | 13.8% | 22.2% | 28.1% | **39.1%** |

平倉時 $|z|$ 中位 **3.82**。未回歸的 **60.9%**，其 $|z|$ 自 4.88 **擴大至 6.61**。

::: {.callout-important}
**這些配對不是尚未回歸，是持續發散。**

延長交易期會讓四成轉盈、六成擴大虧損 → **已否證**。
時間停損亦已否證（4 臂 × 5 門檻 = 20 格全部劣於現行）。
:::

> **機制鏈**：限制候選池 → 選中的配對出樣本收斂性差 → 期末強平實現大額虧損
> → 吃掉收斂交易的獲利。
>
> **分組有害不是因為分錯，而是因為縮小了選擇空間。**

## 形成期的比較為何難以有結論

| 層 | 對照組數 | $SE$ 中位 | **MDE** | 對 0.3 pp 的檢定力 |
| :--- | ---: | ---: | ---: | ---: |
| 形成期（分組層） | 9 | 0.449 | **1.259 pp** | **10%** |
| 交易期（交易層） | 5 | 0.223 | **0.624 pp** | 27% |

**成因是設計，不是資料量：**

| 層 | 兩臂的配對 | 差分消去了什麼 |
| :--- | :--- | :--- |
| 交易期 | **共用同一批** | 市場衝擊 + 配對特異變異 |
| 形成期 | **不同集合** | 僅市場衝擊；標的特異變異殘留 |

標準誤相差 **2.02 倍** → 等效需 **4.1 倍**樣本期間。
欲把 MDE 壓到 0.3 pp：形成期層需 **17.6 倍**樣本（**逾四個世紀**的日資料）。

::: {.callout-important}
**此為結構性限制**：更換分群演算法、增加特徵維度、改良插補方式皆不影響它。
唯一出路是構造使兩臂共用同一批標的的**配對設計**。
:::

# 交易層：門檻選擇 vs. 固定門檻

**設計**：**固定形成期配對**，僅將交易端由固定門檻 Z-Score（$z$=2.0）換成
**DL-THR**（逐期由模型自 9 個動作中選擇門檻）。
配對底涵蓋三種 ML 分群與**傳統 GICS 分組**，以檢驗增益是否依賴特定配對來源。

同參數網格逐格配對檢定（$n = 15$）：

| 配對底 | ΔSharpe | 勝格數 | $p$ | 最佳年化（Z → DL-THR） |
| :--- | :---: | :---: | :---: | :---: |
| **GICS × SSD（傳統）** | **+0.402** | 13/15 | 0.0013 | 1.66% → 2.13% |
| HDBSCAN × SDP | +0.317 | 13/15 | 0.0037 | 1.70% → 2.50% |
| GICS × SDP（傳統） | +0.281 | 12/15 | 0.0070 | 1.65% → 1.97% |
| K-means × SSD | +0.285 | 14/15 | 0.0017 | 1.31% → 1.90% |
| Agglomerative × SSD | +0.263 | **15/15** | 0.0006 | 1.77% → 2.37% |

**五種配對底全部顯著改善**（$p < 0.01$）。
增益最大者為傳統 GICS 分組——**DL-THR 的價值不依賴機器學習分群**。

**穩健性**（15 格中 Sharpe 為正者）：

| 配對底 | Z-Score | → DL-THR |
| :--- | :---: | :---: |
| GICS × SSD | 5/15 | **13/15** |
| HDBSCAN × SDP | 4/15 | 8/15 |
| Agglomerative × SSD | 3/15 | 8/15 |
| K-means × SSD | 2/15 | 5/15 |


## 交易層的四個發現

**一、方向 5/5 為正，校正後 0/5 顯著**
效果量 +0.112 ~ +0.573 pp；增益最大者為**兩條傳統 GICS 配對底**。

**二、三項機械性替代解釋，在 GICS 兩底皆排除**

| 替代解釋 | 檢驗 | 結果 |
| :--- | :--- | :--- |
| SKIP 選股方向 | 置換檢定 | 75 格僅 **4 格**顯著（隨機期望 3.75） |
| 門檻水準 | 同門檻對照 | 僅複製 **0–30%** |
| 總曝險減少 | 隨機跳過同數量 | GICS 兩底 **11/15、10/15** 顯著勝出 |

**三、跨輪訓練變異極小**
五輪獨立重訓的平均 Sharpe 全距 **0.010–0.047**，遠小於增益本身。

**四、價值與分群無關**
增益在傳統 GICS 底最大；K-means 底的增益屬機械效應，Agglomerative 底無增益。

::: {.callout-warning}
三項檢定皆為**事後重抽，不含槽位再配置效應**。
:::

## 反事實標籤值多少錢：DL-THR vs RL-THR

配對底 `Grid (AGG-SSD)`，15 格等權、逐日、循環 block bootstrap（$n$=6,287）。

| 對照 | 年化Δ | 95% CI (pp) | $p$ |
| :--- | ---: | :---: | ---: |
| **DL-THR − Z-Score**（參照） | **+0.787 pp** | [+0.34, +1.27] | **0.0011** |
| RL-THR ($\varepsilon$=0.05) − Z-Score | +0.707 pp | [+0.10, +1.45] | **0.0400** |
| **RL-THR ($\varepsilon$=0.10) − Z-Score** | **+0.802 pp** | [+0.20, +1.54] | **0.0191** |
| RL-THR ($\varepsilon$=0.20→0.02) − Z-Score | +0.669 pp | [+0.08, +1.38] | **0.0450** |
| RL-THR ($\varepsilon$=0.05) − DL-THR | −0.081 pp | [−0.47, +0.34] | 0.6935 |
| **RL-THR ($\varepsilon$=0.10) − DL-THR** | **+0.015 pp** | **[−0.38, +0.45]** | **0.9409** |
| RL-THR ($\varepsilon$=0.20→0.02) − DL-THR | −0.118 pp | [−0.49, +0.28] | 0.5357 |

最有利排程下兩臂相差 **+0.015 pp**，區間 **[−0.38, +0.45]** 幾乎對稱橫跨零；
三組 $\varepsilon$ 一致（−0.12 ~ +0.02 pp）。兩端皆遠小於總增益 +0.787 pp
→ 這是「兩端皆小」的相當，不是檢定力不足。

::: {.callout-warning}

### 兩個報告口徑給出相反的答案

改報**最佳格**（五輪獨立重跑）：DL-THR 中位年化 **2.387%**［2.341, 2.649］
vs RL-THR $\varepsilon$=0.10 的 **2.107%**［1.985, 2.192］——
**五輪全勝且全距完全不重疊**。

不矛盾：best-of-15 放大微小而系統性的優勢（資訊量九倍的一臂更**可靠地**產出好的
極大值），等權組合把這個選擇效應平均掉。本研究一律以等權為口徑，故結論取「相當」。

**這是報告口徑重要性的具體實例：同一份資料在最佳格口徑下會支持相反的結論。**

:::

**限制**　差距同時含「資訊量僅 1/9」與「探索成本」，本設計無法分離——不探索就沒有樣本。


## 交易層 的重訓穩健性（五輪獨立訓練）

DL-THR 網路未固定隨機種子，每輪重新訓練。下表為五輪各自取網格最佳後的跨輪統計。

| 配對底 | 最佳年化 中位［範圍］ | 最佳 Sharpe 中位［範圍］ | 正 Sharpe 最少 |
| :--- | :---: | :---: | :---: |
| Agglomerative × SSD | 2.39%［2.34, 2.65］ | 0.34［0.34, 0.38］ | 6/15 |
| HDBSCAN × SSD-DTW-PCA | **2.46%**［2.41, 2.54］ | 0.31［0.31, 0.32］ | **8/15** |
| K-means × SSD | 1.73%［1.53, 1.90］ | 0.26［0.23, 0.28］ | 5/15 |

::: {.callout-important}

### 為何必須報告變異數

Agglomerative 底的單次訓練值為 **2.65%**，恰為其五輪範圍的**上界**；
其中位數 2.39% 低於 HDBSCAN 底的 2.46%。
若僅報告單次結果，將得出「Agglomerative 底的 DL-THR 表現最佳」之結論，
而該結論**無法在重訓下複現**。

跨策略比較一律以中位數為準；範圍寬度本身亦為一項結果指標
（反映該配對底提供的學習訊號穩定性）。

:::


## DL-THR 學到了什麼：決策行為解析

動作選擇未落庫，但可由 `trade_logs` 完整還原
（SKIP → 全期 `HOLD_CASH (SKIP)`；entry_z → 進場列的 $\min|Z|$）。
下表取每策略 TOP N 最大之網格（樣本最充足）。

**決策分布**（動作選單 $entry_z \in \{1.5, 2.0, 2.5, 3.0\}$，靜態基準 2.0）：

| 配對底 | 配對期數 | SKIP 率 | 門檻中位 | 1.5 / 2.0 / 2.5 / 3.0 | 偏離基準 |
| :--- | :---: | :---: | :---: | :---: | :---: |
| Agglomerative | 1462 | 35.0% | 2.22 | 12 / 40 / 21 / 27 % | **61%** |
| HDBSCAN | 1475 | 37.1% | 2.27 | 8 / 40 / 22 / 29 % | 62% |
| K-means | 1377 | 38.7% | 2.20 | 15 / 41 / 19 / 26 % | 62% |

**增益來源分解**（DL-THR − Z-Score 總損益差，逐配對拆解）：

| 配對底 | 總增益 | SKIP 貢獻 | 門檻貢獻 |
| :--- | :---: | :---: | :---: |
| Agglomerative | 3106 | 1117（36%） | **1989（64%）** |
| HDBSCAN | 845 | 260（31%） | **586（69%）** |
| K-means | 1361 | 711（52%） | 650（48%） |

::: {.callout-warning}

### SKIP 並非選擇性技巧

被 DL-THR 跳過的配對，其在 Z-Score 端的虧損比例與留下者**幾乎相同**
（Agglomerative 60% vs 60%；HDBSCAN 48% vs 51%；K-means 50% vs 48%），
Mann-Whitney 檢定 $p$ = 0.35 ~ 0.93，無一顯著。

「被跳過者損益為負」不足以證明技巧——配對平均損益本就為負，
**隨機棄權同樣會「避開虧損」**。判準須為其損益是否顯著低於留下者。

:::

**結論**：DL-THR 的價值在於**為每組配對挑選適當的進出場門檻**（62% 的決策偏離靜態基準），
而非拒絕交易。此結果在獨立資料上支持 Kim & Kim (2019) 的門檻最適化主張，
並量化了該機制的貢獻比重。

**可解釋性的邊界**：門檻選擇與排序名次的 Spearman $\rho$ 僅 −0.03 ~ +0.07，
模型並非依循「名次差則提高門檻」之類的簡單規則，
其決策為 12 維形成期特徵的非線性組合，本研究未能進一步歸因。


## 交易層的假設檢定：相對主張

| 配對底 | 年化Δ(pp) | 95% CI (pp) | IR | $p$ | BH 校正 $p$ |
| :--- | ---: | :---: | ---: | ---: | ---: |
| **GICS × SSD（傳統）** | **+0.573** | [+0.09, +1.19] | 0.427 | 0.042 | 0.087 |
| **GICS × SDP（傳統）** | **+0.495** | [+0.08, +0.96] | 0.357 | 0.028 | 0.087 |
| HDBSCAN × SDP | +0.456 | [+0.04, +0.96] | 0.344 | 0.052 | 0.087 |
| K-means × SSD | +0.283 | [−0.04, +0.58] | 0.223 | 0.073 | 0.091 |
| Agglomerative × SSD | +0.112 | [−0.20, +0.44] | 0.086 | 0.488 | 0.488 |

**方向 5/5 為正，校正前 2/5 顯著，BH 校正後 0/5。**

::: {.callout-important}
觀測效果中位 **+0.456 pp** 對最小可偵測效果 **0.624 pp**
→ 屬「效果存在但接近偵測極限」，非「檢定力不足到無從判斷」。
:::

> 增益最大者為**傳統 GICS 配對底** → 門檻選擇的改良與「配對如何找到」**正交**。

## 絕對績效：全部低於無風險利率

主軸 15 格的等權年化（15 個參數配置等權，無選擇偏誤）：

| 分組 | SSD | DTW | SSD-DTW-PCA |
| :--- | ---: | ---: | ---: |
| **不分組** | +0.715% | **+0.812%** | +0.521% |
| GICS 產業 | +0.346% | +0.415% | +0.495% |
| Agglomerative | −0.112% | +0.199% | +0.059% |
| HDBSCAN | −0.391% | +0.049% | −0.215% |
| K-means | −0.583% | −0.499% | −0.511% |

疊加 DL-THR 後最佳者為 **GICS-SDP**：年化 **+0.914%**、25 年終值 **12,548**、
動用資本口徑 +2.161%、平均 Sharpe 0.217。

::: {.callout-important}
**同期僅持有無風險資產（2% 假設）將得約 16,400。**

本研究的任何配置皆未達此水準。
:::

**Deflated Sharpe**（$N$=53、$SR_0$=0.37）：最高 **0.900**（形成窗 504 對照臂），**無一通過 0.95**；該臂後半期 Sharpe 為 −0.395。

## 相對比較與絕對績效是兩件事

| | 逐日差分 bootstrap | 絕對 bootstrap |
| :--- | :--- | :--- |
| $H_0$ | 兩臂績效相同 | 策略平均日報酬為零 |
| 對照物 | 同配對、同參數格的另一臂 | **零** |
| 主張性質 | **相對** | **絕對** |

配對設計消去共同的市場風險 → 訊噪比大幅提高；
絕對檢定沒有對照物可消噪，其標準誤必然大得多。

::: {.callout-important}
本研究的三項檢定**全部是相對比較**。即使某一項達顯著，
也只代表「A 優於 B」，不代表 A 本身可獲利。

而絕對績效**明確為負**（低於無風險利率），
故連「相對較優者是否值得部署」都不成立。
:::

> 本研究的定位因此清楚：**方法論研究**——
> 回答「哪一層的改良可被驗證、哪一層不能」，而非提出一個可交易的策略。

# 兩層組合：實務上要部署的那個檢定

> **組合系統**　資料驅動分群 ＋ **DL-THR 門檻選擇**
> **傳統基準**　GICS 產業分組 ＋ **固定門檻 Z-Score**（排序已對齊）

| 期間 | 分群法 | 年化Δ(pp) | 95% CI (pp) | $p$ | BH 校正 $p$ |
| :--- | :--- | ---: | :---: | ---: | ---: |
| 全期 | Agglomerative | **−0.334** | [−1.28, +0.53] | 0.469 | 0.558 |
| 全期 | HDBSCAN | **−0.265** | [−1.20, +0.59] | 0.558 | 0.558 |
| 全期 | K-means | **−0.630** | [−1.73, +0.36] | 0.228 | 0.558 |
| 2012+ | Agglomerative | +0.470 | [−0.36, +1.21] | 0.234 | 0.350 |
| 2012+ | HDBSCAN | +0.193 | [−0.53, +0.84] | 0.587 | 0.587 |
| 2012+ | K-means | +0.582 | [−0.21, +1.31] | 0.135 | 0.350 |

**成分分解**（全期）：DL-THR 成分 **+0.112 ~ +0.456**（皆正）、
分群成分 **−0.447 ~ −0.913**（皆負且量級更大）。

::: {.callout-important}
**完整系統劣於傳統基準，因為分群層的損害超過交易端的貢獻。**
兩個成分並不互補——若要保留交易端的改良，應搭配傳統產業分組或不分組。
:::

## 交易層 的 Regime 穩健性

以等權市場的 63 日滾動波動率三分位與 126 日趨勢標記每個交易日，
分層計算年化 Sharpe。檢驗增益是否僅來自特定市場環境。

口徑同全篇：**15 格等權、逐格對齊兩臂** → 唯一變因仍是交易端。

| 配對底 / 交易端 | Calm | Normal | Turbulent | Bull | Bear |
| :--- | :---: | :---: | :---: | :---: | :---: |
| Agglomerative × SSD | −0.47 | −0.66 | 0.55 | −0.07 | 0.03 |
| Agglomerative **+ DL-THR** | **−0.23** | **−0.43** | 0.31 | **+0.02** | 0.00 |
| HDBSCAN × SDP | −0.42 | −0.85 | 0.52 | −0.20 | 0.13 |
| HDBSCAN **+ DL-THR** | **−0.19** | **−0.45** | 0.44 | **−0.05** | **0.25** |
| K-means × SSD | −0.45 | −0.90 | 0.26 | −0.30 | −0.21 |
| K-means **+ DL-THR** | **−0.06** | **−0.55** | 0.11 | **−0.10** | **−0.13** |
| GICS × SSD | −0.41 | −0.50 | 0.77 | −0.10 | 0.46 |
| GICS × SSD **+ DL-THR** | **−0.23** | **−0.26** | 0.75 | **+0.04** | **0.58** |
| GICS × SDP | −0.25 | −0.47 | 0.76 | −0.02 | 0.45 |
| GICS × SDP **+ DL-THR** | **−0.05** | **−0.15** | 0.68 | **+0.14** | **0.50** |

**DL-THR 在 25 個 regime 格中改善 19 格、劣化 6 格。**
劣化的 6 格中有 5 格落在 Turbulent——即原本就獲利的那一檔；
改善集中於虧損較大的 Calm 與 Normal。

**同時揭露一項策略層面的限制**：配對交易的獲利完全集中於
**高波動與空頭**環境；**平靜期十組配置全部為負 Sharpe**（−0.47 ~ −0.05），
DL-THR 只能減輕虧損而無法反轉。
此與本研究另一項發現一致：損益集中於高分散度期間（附錄 D 的 regime 閘門依據）。


## 成本敏感度：Break-even 分析

成本模型可解析求解——進出場費用 = friction × 名目額，
且每配對名目額恰等於每配對資金，故往返 break-even
$c^* = 2\,(0.29\% + 	ext{淨利} / \Sigma	ext{名目額})$。

| 配對底 | Z-Score | → DL-THR | 差（bps） | Z 餘裕 | DL 餘裕 |
| :--- | :---: | :---: | :---: | :---: | :---: |
| Agglomerative × SSD | 0.555% | **0.567%** | +1.2 | **−2.5** | **−1.3** |
| HDBSCAN × SDP | 0.519% | **0.690%** | +17.0 | **−6.1** | +11.0 |
| K-means × SSD | 0.391% | **0.444%** | +5.3 | **−18.9** | **−13.6** |
| GICS × SSD | 0.676% | **0.920%** | +24.4 | +9.6 | **+34.0** |
| GICS × SDP | 0.711% | **0.924%** | +21.3 | +13.1 | **+34.4** |

*往返 break-even 成本；現行假設 0.58%（單邊 0.29%，Do & Faff 2012）。
口徑：15 格等權、逐格對齊兩臂*

**DL-THR 在五種配對底上一致提高成本承受度**（+1.2 ~ +24.4 bps）。機制為 SKIP 與
較高的進場門檻共同減少交易次數，使同樣的毛利分攤在更少的名目額上。

::: {.callout-important}

### 但餘裕的離散程度遠大於中心水準

十組配置的餘裕介於 **−18.9 ~ +34.4 bps**，全距 53 bps，
幾與成本假設本身（58 bps）同量級。

**三個資料驅動分群底的 Z-Score 臂已經為負**——現行假設下它們本來就不獲利，
且 DL-THR 只把其中一條（HDBSCAN）推回正值。
兩條 GICS 底加 DL-THR 則達 +34 bps，在成本維度上站得住。

**成本可行性取決於配對底，不是全篇一致的結論。**

:::


# 附錄摘要

主軸之外的支撐性實驗，完整數據見 `results/result.db` 與
`archive/config_archived_strategies.py`。

| 附錄 | 內容 | 主要結果 |
| :--- | :--- | :--- |
| **A** | 文獻原始設定復現 | Gatev (2006) 原型、許鈞翔 (2025) ADF 0.01 設定；Grid (GICS-SSD) 與原生 SSD Rolling 數值完全相同 |
| **A′** | 前行研究差異的受控定位 | 與許鈞翔 (2025) 的八項差異清單；隔離「進場時點」一項：回歸式進場使三種排序年化全部下降 −0.77 ~ −0.91pp，BH 後 0/3 顯著（`analysis/hsu25_entry_timing.py`） |
| **B** | 篩選消融（有/無三道統計篩選） | 篩選貢獻 +0.25 ~ +0.87pp，三種排序下皆為正 |
| **C** | 特徵層消融 | 多尺度動量（−0.94 ~ −2.43pp）、SEC 結構性財報比率（±0.22pp 噪音範圍）皆無助益 |
| **D** | 延伸探索：regime 條件化進場 | 低分散度閘門使全網格 Sharpe 轉正、MDD 下降；三層疊加五輪中位 2.69%［2.65, 2.71］、最差輪 14/15 正 Sharpe |
| **I** | Regime／成本可重現腳本 | `analysis/regime_cost_dsr_eval.py`：regime 分層 Sharpe、break-even 成本、DSR |
| **H** | 粒度掃描可重現腳本 | `analysis/granularity_sweep.py`：切割門檻 × 候選池 × 跨產業比例 × 績效 |
| **G** | 行為解析可重現腳本 | `analysis/drl_behavior.py`：決策分布／SKIP 技巧性／增益來源分解 |
| **F** | 統計檢定可重現腳本 | `analysis/proposition2_stats.py`：配對 t／Wilcoxon／逐輪檢定／Newey-West／DSR 四項一次產出 |
| **E** | 參數敏感性 | ADF 門檻 0.01 / 0.05 / 0.1 對照，說明本研究採 0.05 的實證依據 |
| **J** | 反事實標籤的價值 | `analysis/prop2_label_information.py`：RL-THR（部分回饋 + ε-greedy）vs DL-THR（全資訊監督），等權口徑下相當 |


## 附錄 B、E 的方法論意涵（值得在正文引用）

**篩選是必要的**：純距離排序不足以識別可交易配對——距離度量回答「歷史走勢多接近」，
共整合檢定回答「價差是否會回歸」，兩者結合才構成有效選取（SSD 排序下 0.79% → 1.66%）。

**ADF 門檻 0.05 優於文獻慣用的 0.01**：實證顯示過嚴門檻反而降低績效
（Agglomerative FMP：1.27% @0.01 vs 1.77% @0.05），且 0.05 → 0.1 已飽和。
機制為「排序流程先按距離排序、再逐一檢定並填滿 top_n」——門檻收緊迫使系統
往距離更遠的候選尋找，**以經濟相似性換取統計顯著性，淨效果為負**。
候選池充足時不再出現早期「篩選過嚴導致無配對」的問題（ADF 0.01 仍可填滿 20/20 對）。


# 結論與限制

| 檢定 | 方向 | 校正後顯著 |
| :--- | :--- | :---: |
| 分組層 | 9/9 偏向 GICS | **0/9** |
| 交易層 | 5/5 為正 | **0/5** |
| 兩層組合 | 全期 3/3 為負 | **0/3** |

::: {.callout-important}

**其一，分組層是淨損害**，限制愈強損害愈大（不分組 > GICS > 任一分群，15 格逐格成立）。
機制為期末強制平倉：強平率 36.8% → 43.1%，與績效的相關在 15 個配置中一致為負。
強平的配對再追一個交易期僅 **39.1%** 回歸——**是持續發散，非尚未回歸**。

**其二，門檻選擇是唯一方向為正的改良**，價值與分群無關；
三項機械性解釋在 GICS 兩底皆排除。惟量級接近偵測極限，校正後不顯著。

**其三，兩層的檢定力不對等來自設計**：形成期層需 **4.1 倍**樣本。
此為結構性限制，更換演算法或增加特徵皆不影響。

:::

::: {.callout-warning}
### 絕對績效與定位

最佳配置 25 年將 10,000 變為 **12,548**，低於無風險利率假設的約 **16,400**；
DSR 無一通過 0.95。

**本研究不主張任何配置具可交易的獲利能力；全部結論均為相對宣稱。**
:::

> **主要限制**：母體僅 S&P 500（約 600 檔，動機來源使用 CRSP 數千檔）、
> 特徵維度受限（2008 年前無基本面資料）、
> 形成期層的檢定力不足且無法以更多資料補救。